# Query the example OGC Records with DuckDB

This notebook follows the complete serialization and query workflow:

1. Read the supplied OGC Records JSON files.
2. Write their queryable fields and PySTAC-serialized values to Parquet.
3. Read collection descriptions and select rows by persisted membership.
4. Translate CQL2 JSON filters into parameterized DuckDB SQL.
5. Reconstruct complete OGC Records from the matching rows.

The output is `notebooks/output/ogc-records.parquet`.
Query execution stays outside the serialization library.

All supplied geometries are null, so the examples use attribute filters.
They do not require DuckDB's spatial extension.

## 1. Prepare the notebook environment

Use Python 3.12–3.14 and install the optional notebook dependencies from the repository root:

```bash
pip install -e ".[notebook]"
jupyter lab
```

Select the kernel for that environment.
The first cell checks that the backend is exactly **pygeofilter-duckdb==0.2.0**
and prints the installed package versions.

The repository lookup supports launching Jupyter from either the repository root
or the `notebooks` directory.

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path

import duckdb
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import JSON, display
from pystac.extensions.ogc_record import OGCRecord
from pystac import ItemCollection
from pygeofilter.parsers.cql2_json import parse
from pygeofilter_duckdb.evaluate import to_sql_where_params

from ogc_records_parquet import (
    JSONObject,
    arrow_to_records,
    read_collection_ids,
    read_collections,
    read_parquet,
    records_to_arrow,
    to_parquet,
)

# Keep the query backend consistent with the version used by this example.
assert version("pygeofilter-duckdb") == "0.2.0"
print(
    {
        name: version(name)
        for name in ("ogc-records-parquet", "pygeofilter-duckdb", "duckdb", "pyarrow")
    }
)

# Find the example directory even when Jupyter starts inside notebooks/.
repository = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "examples" / "ogc-records").is_dir()
)

## 2. Write and verify the Parquet file

Read the Records from `examples/ogc-records/*/records/` in sorted path order.
The `hazard-mapping` collection contains four Records;
`earth-observation-processing` contains seven.
Each collection has a separate `collection.json` metadata file,
which is stored in the Parquet library metadata and verified on readback.
This demonstration keeps the source Records in a list for comparison;
it is not a bounded-memory ingestion example.

`records_to_arrow` projects the default query fields and stores each original
Record alongside its WKB geometry and derived bbox columns.
`to_parquet` writes the Arrow batches to the output file.
Running this cell again replaces that file.

The assertion then reads the Parquet back and compares all Records with their
PySTAC-serialized values. The displayed schema shows the available columns and metadata.

The ingestion cell explicitly pairs each Record with its source collection directory.
It passes these associations as `collection_ids`, independently of Record JSON,
and verifies the 4/7 split by reading the stored `__collection_id` column.

In [ ]:
example_paths = sorted((repository / "examples" / "ogc-records").glob("*/records/*.json"))

# Materialize this small example set so later checks can compare original values.
source_rows = [(OGCRecord.from_file(path), path.parent.parent.name) for path in example_paths]
records = [record for record, _ in source_rows]
collection_ids = [identifier for _, identifier in source_rows]
assert records

# Collection metadata is separate from each unchanged Record document.
collection_paths = sorted((repository / "examples" / "ogc-records").glob("*/collection.json"))
collections = {
    path.parent.name: json.loads(path.read_text(encoding="utf-8")) for path in collection_paths
}
assert set(collections) == {"hazard-mapping", "earth-observation-processing"}

output_directory = repository / "notebooks" / "output"
output_directory.mkdir(parents=True, exist_ok=True)
parquet_path = output_directory / "ogc-records.parquet"

# Small batches demonstrate writing several Arrow batches into one Parquet file.
to_parquet(
    records_to_arrow(records, batch_size=4, collections=collections, collection_ids=collection_ids),
    parquet_path,
)

# Compare complete JSON values, including fields absent from query projections.
assert [record.to_dict(transform_hrefs=False) for record in list(read_parquet(parquet_path))] == [
    record.to_dict(transform_hrefs=False) for record in records
]
assert read_collections(parquet_path) == collections
assert list(read_collection_ids(parquet_path)) == collection_ids

# Inspect persisted membership independently of the original Record documents.
stored_ids = pq.read_table(parquet_path, columns=["__collection_id"])["__collection_id"].to_pylist()
assert stored_ids == collection_ids
assert {identifier: stored_ids.count(identifier) for identifier in collections} == {
    "hazard-mapping": 4,
    "earth-observation-processing": 7,
}
print("Verified persisted membership: hazard-mapping=4, earth-observation-processing=7")
print(f"Wrote and verified {len(records)} Records: {parquet_path}")

# Retain the file metadata for reconstruction after querying through DuckDB.
stored_schema = pq.read_schema(parquet_path)
display(stored_schema)

## 3. Read collections and select their Records

Collection descriptions and row membership are separate:

- `read_collections` retrieves the two descriptions from file metadata.
- `read_collection_ids` retrieves one storage membership value per row.
- The Arrow `__collection_id` column selects the rows belonging to a collection.

Membership was supplied explicitly during ingestion. Neither descriptions nor
Record identifiers assign membership. Duplicate Record IDs can belong to different
collections, so selection uses the membership column.

The following cell reads everything from the written Parquet file. It verifies
four hazard-mapping Records and seven earth-observation-processing Records,
then displays both collection descriptions. Null membership would mean unknown;
legacy format 1.0.0 files return null membership for every row.

In [ ]:
stored_collections = read_collections(parquet_path)
stored_memberships = list(read_collection_ids(parquet_path))
stored_table = pq.read_table(parquet_path)

assert stored_collections == collections
assert stored_memberships == collection_ids

collection_sizes = {"hazard-mapping": 4, "earth-observation-processing": 7}

for collection_id, expected_size in collection_sizes.items():
    # Filter by the persisted association, independently of the original JSON payload.
    selected_table = stored_table.filter(
        pc.equal(stored_table.column("__collection_id"), pa.scalar(collection_id))
    )
    selected_records = list(arrow_to_records(selected_table))

    # Directory associations are used only as independent source evidence in this check.
    expected_records = [
        record for record, source_collection in source_rows if source_collection == collection_id
    ]
    assert selected_table.num_rows == expected_size
    assert [record.to_dict(transform_hrefs=False) for record in selected_records] == [
        record.to_dict(transform_hrefs=False) for record in expected_records
    ]
    assert all("__collection_id" not in record.extra_fields for record in selected_records)

    print(f"{collection_id}: {len(selected_records)} original Records")
    display(JSON(stored_collections[collection_id]))

## 4. Map filter properties and reconstruct results

The Arrow schema contains literal dotted column names such as `properties.title`.
The field mapping lets a CQL2 filter refer to that column as `title`.
Only names in this mapping are available to the query helper.

The complete Record is stored in `__record_json`.
This stores `OGCRecord.to_dict()` output, including nested objects and extension fields.
The extension normalizes metadata timestamps to UTC whole seconds and some optional members.
The round-trip check compares PySTAC serialization, not the raw source JSON.

The query helper performs three operations:

1. Parse a CQL2 JSON object and compile it into a SQL predicate and bound parameters.
2. Execute the query against the Parquet-backed `records` view.
3. Restore the stored library metadata and decode the original Records.

DuckDB does not retain that library metadata in its Arrow query result.
Restoring it lets `arrow_to_records` identify the original JSON payload column.
Results are ordered by their string identifier to make comparisons reproducible.

The `collectionId` filter name maps to storage-only `__collection_id`.
It is available for filtering without adding a collection property to Record JSON.

In [ ]:
connection = duckdb.connect()
connection.read_parquet(str(parquet_path)).create_view("records")

# Map public filter names to complete, literal Parquet column names.
field_mapping = {
    "id": "id",
    "collectionId": "__collection_id",
    "type": "type",
    "title": "properties.title",
    "description": "properties.description",
    "resourceType": "properties.type",
    "created": "properties.created",
    "updated": "properties.updated",
}


def query_records(filter_json: JSONObject) -> list[OGCRecord]:
    """Return original Records matching a CQL2 JSON attribute filter.

    Args:
        filter_json: CQL2 JSON expression using names in the field mapping.

    Returns:
        Matching Records ordered by their string identifier.

    Raises:
        KeyError: If a filter references an unmapped property.
    """
    # Bind literal values separately; the backend quotes mapped identifiers.
    predicate, parameters = to_sql_where_params(parse(filter_json), field_mapping)
    print("SQL predicate:", predicate)
    print("Bound parameters:", parameters)

    result = connection.execute(
        f"SELECT * FROM records WHERE {predicate} ORDER BY id", parameters
    ).to_arrow_table()

    # Reconstruct from the original payload rather than the flattened projections.
    restored = result.replace_schema_metadata(stored_schema.metadata)
    return list(arrow_to_records(restored))

## 5. Query collections with CQL2 JSON

Use an equality expression on `collectionId` to retrieve a collection's Records.
The helper compiles this into a parameterized predicate on `__collection_id`.
The description mapping remains independent and is unchanged by filtering.

The checks below select both collections, compare full Record JSON values with
the source examples, and confirm that an unknown collection returns no Records.
An `and` expression then combines collection membership with the flood-title filter.

In [ ]:
for collection_id, expected_size in collection_sizes.items():
    collection_filter = {
        "op": "=",
        "args": [{"property": "collectionId"}, collection_id],
    }
    collection_records = query_records(collection_filter)

    expected_records = sorted(
        [record for record, source_collection in source_rows if source_collection == collection_id],
        key=lambda record: str(record.id),
    )
    assert len(collection_records) == expected_size
    assert [record.to_dict(transform_hrefs=False) for record in collection_records] == [
        record.to_dict(transform_hrefs=False) for record in expected_records
    ]

    print(f"CQL2 collection query: {collection_id}, {len(collection_records)} Records")
    display(JSON(ItemCollection(collection_records).to_dict()))

assert (
    query_records({"op": "=", "args": [{"property": "collectionId"}, "unknown-collection"]}) == []
)

hazard_flood_records = query_records(
    {
        "op": "and",
        "args": [
            {"op": "=", "args": [{"property": "collectionId"}, "hazard-mapping"]},
            {"op": "like", "args": [{"property": "title"}, "%Flood%"]},
        ],
    }
)
assert [record.to_dict(transform_hrefs=False) for record in hazard_flood_records] == [
    record.to_dict(transform_hrefs=False)
    for record in sorted(
        [
            record
            for record, source_collection in source_rows
            if source_collection == "hazard-mapping" and "Flood" in record.title
        ],
        key=lambda record: str(record.id),
    )
]
assert read_collections(parquet_path) == stored_collections

## 6. Find Records with a flood-related title

A CQL2 JSON expression uses `op` for the operator and `args` for its operands.
`{"property": "title"}` refers to a mapped field; `"%Flood%"` is a literal pattern.
The `%` wildcard matches any sequence of characters.

This `like` filter is case-sensitive and matches two supplied Records:
Diachronic Flood Detection and Flood Depth Mapping.
The result is displayed inside a GeoJSON `FeatureCollection` envelope.
The comparison checks that each returned Record retains all its original fields.

In [ ]:
filter_json = {"op": "like", "args": [{"property": "title"}, "%Flood%"]}

matching_records = query_records(filter_json)

# Independently check both membership and complete Record contents.
expected_records = sorted(
    [record for record in records if "Flood" in record.title],
    key=lambda record: str(record.id),
)
assert [record.to_dict(transform_hrefs=False) for record in matching_records] == [
    record.to_dict(transform_hrefs=False) for record in expected_records
]
assert matching_records

print(f"Matched {len(matching_records)} Records")
display(JSON(ItemCollection(matching_records).to_dict()))

## 7. Combine filters and check empty results

The `and` operator takes two nested expressions: an identifier equality test
and the title pattern from the previous example.
Together they select the Diachronic Flood Detection Record.

The remaining checks cover two cases:

- An unknown identifier returns an empty list.
- An `or` filter returns the flood-title matches plus the Burned Area Severity Record.

Each result is compared with the original Records, then the DuckDB connection is closed.

In [ ]:
record_identifier = "diachronic-flood-detection"

single_record = query_records(
    {
        "op": "and",
        "args": [
            {"op": "=", "args": [{"property": "id"}, record_identifier]},
            {"op": "like", "args": [{"property": "title"}, "%Flood%"]},
        ],
    }
)

# Both conditions must hold, and the complete original Record must be returned.
assert [record.to_dict(transform_hrefs=False) for record in single_record] == [
    record.to_dict(transform_hrefs=False)
    for record in [record for record in records if record.id == record_identifier]
]
assert len(single_record) == 1
display(JSON(single_record[0].to_dict()))

# A valid filter with no matches returns an empty list.
assert query_records({"op": "=", "args": [{"property": "id"}, "does-not-exist"]}) == []

# OR returns the union of matching Records without duplicating a row.
assert [
    record.to_dict(transform_hrefs=False)
    for record in query_records(
        {
            "op": "or",
            "args": [
                {"op": "like", "args": [{"property": "title"}, "%Flood%"]},
                {"op": "=", "args": [{"property": "id"}, "burned-area-severity"]},
            ],
        }
    )
] == [
    record.to_dict(transform_hrefs=False)
    for record in sorted(
        [
            record
            for record in records
            if "Flood" in record.title or record.id == "burned-area-severity"
        ],
        key=lambda record: str(record.id),
    )
]

print("Verified LIKE, equality, AND, OR, empty results, and complete JSON reconstruction.")

connection.close()